# Gold — executive_kpis_daily

Build satu tabel Gold sesuai kontrak `gold.sql`.

**Prerequisite: gold.order_360, gold.customer_daily, dan gold.product_daily sudah tersedia.**

Jalankan notebook dari atas ke bawah.

In [ ]:
import os
from datetime import datetime, timezone
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
DB_URL = os.getenv("LOCAL_DATABASE_URL")
if DB_URL and DB_URL.startswith("postgresql://"):
    DB_URL = DB_URL.replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(DB_URL)
pipeline_run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

print("Database connection berhasil")
print("Pipeline run ID:", pipeline_run_id)


## 1. Build `executive_kpis_daily`

In [ ]:
customer_daily = pd.read_sql("SELECT * FROM gold.customer_daily", engine)
product_daily = pd.read_sql("SELECT * FROM gold.product_daily", engine)
order_360 = pd.read_sql("SELECT * FROM gold.order_360", engine)
order_360.to_sql("_tmp_order_360", engine, if_exists="replace", index=False)
customer_daily.to_sql("_tmp_customer_daily", engine, if_exists="replace", index=False)
product_daily.to_sql("_tmp_product_daily", engine, if_exists="replace", index=False)
customer_daily.to_sql("_tmp_customer_daily", engine, if_exists="replace", index=False)
product_daily.to_sql("_tmp_product_daily", engine, if_exists="replace", index=False)

executive_kpis_daily = pd.read_sql("""
WITH orders AS (
    SELECT order_date metric_date, COUNT(*) total_orders, SUM(gross_merchandise_value) gross_revenue, SUM(net_revenue) net_revenue,
           AVG(gross_merchandise_value) average_order_value,
           COUNT(*) FILTER (WHERE refunded_amount>0) refunded_orders,
           COUNT(*) FILTER (WHERE payment_status='PAYMENT_CAPTURED') captured_orders,
           COUNT(*) FILTER (WHERE return_status<>'NO_RETURN') returned_orders,
           COUNT(*) FILTER (WHERE return_status<>'NO_RETURN' OR order_status IN ('COMPLETED','RETURNED','CANCELLED')) resolved_orders
    FROM _tmp_order_360 GROUP BY order_date
),
cust AS (
    SELECT metric_date, COUNT(DISTINCT customer_id) active_customers, AVG(CASE WHEN repeat_customer_flag THEN 1.0 ELSE 0.0 END) repeat_customer_rate,
           SUM(support_contacts) support_contacts, SUM(order_count) customer_orders
    FROM _tmp_customer_daily GROUP BY metric_date
),
stock AS (
    SELECT metric_date, AVG(CASE WHEN stockout_flag THEN 1.0 ELSE 0.0 END) stockout_rate
    FROM _tmp_product_daily WHERE stockout_flag IS NOT NULL GROUP BY metric_date
)
SELECT o.metric_date,o.total_orders,o.gross_revenue,o.net_revenue,o.average_order_value,
       CASE WHEN o.captured_orders=0 THEN 0 ELSE o.refunded_orders::numeric/o.captured_orders END refund_rate,
       CASE WHEN o.resolved_orders=0 THEN 0 ELSE o.returned_orders::numeric/o.resolved_orders END return_rate,
       COALESCE(c.repeat_customer_rate,0) repeat_customer_rate, COALESCE(s.stockout_rate,0) stockout_rate,
       COALESCE(c.active_customers,0) active_customers,
       CASE WHEN o.total_orders=0 THEN 0 ELSE COALESCE(c.support_contacts,0)::numeric/o.total_orders END support_contact_rate,
       NOW() AT TIME ZONE 'UTC' data_freshness_utc
FROM orders o LEFT JOIN cust c USING(metric_date) LEFT JOIN stock s USING(metric_date)
ORDER BY o.metric_date
""", engine)
print("executive_kpis_daily:", len(executive_kpis_daily), "rows")


## 2. Contract types + pipeline_run_id

In [ ]:
executive_kpis_daily["pipeline_run_id"] = pipeline_run_id


## 3. Recreate target table sesuai `gold.sql`

In [ ]:
gold_ddl = """CREATE SCHEMA IF NOT EXISTS gold;
DROP TABLE IF EXISTS gold.executive_kpis_daily;
CREATE TABLE gold.executive_kpis_daily (
    metric_date DATE PRIMARY KEY,
    total_orders INTEGER NOT NULL,
    gross_revenue NUMERIC(14,2) NOT NULL,
    net_revenue NUMERIC(14,2) NOT NULL,
    average_order_value NUMERIC(14,2) NOT NULL,
    refund_rate NUMERIC(14,4) NOT NULL,
    return_rate NUMERIC(14,4) NOT NULL,
    repeat_customer_rate NUMERIC(14,4) NOT NULL,
    stockout_rate NUMERIC(14,4) NOT NULL,
    active_customers INTEGER NOT NULL,
    support_contact_rate NUMERIC(14,4) NOT NULL,
    data_freshness_utc TIMESTAMPTZ NOT NULL,
    pipeline_run_id TEXT NOT NULL
);"""
with engine.begin() as conn:
    conn.execute(text(gold_ddl))

executive_kpis_daily.to_sql("executive_kpis_daily", engine, schema="gold", if_exists="append", index=False, method="multi")
print("gold.executive_kpis_daily berhasil dibuat.")

## 4. Final validation

In [ ]:
df = pd.read_sql("SELECT * FROM gold.executive_kpis_daily", engine)
print("Rows:", len(df))
print("Unique dates:", df["metric_date"].nunique())
print("Duplicate rows:", len(df) - len(df.drop_duplicates()))
print("NULL values:", int(df.isna().sum().sum()))
